In [1]:
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', 'scipy', 'tqdm', 'lightgbm', '-q'])

CompletedProcess(args=['c:\\Users\\mhimi\\anaconda3\\python.exe', '-m', 'pip', 'install', 'scipy', 'tqdm', 'lightgbm', '-q'], returncode=0)

In [2]:
import pandas as pd
import numpy as np
import json
import gc
import os
import time
from tqdm import tqdm
from scipy.signal import butter, sosfiltfilt
import lightgbm as lgb
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import LeaveOneGroupOut
from sklearn.metrics import log_loss

print('All imports OK')

All imports OK


In [3]:
df = pd.read_csv('train.csv')
print(df.shape)
print(df.head())

(4867421, 28)
   crew experiment      time  seat   eeg_fp1     eeg_f7     eeg_f8     eeg_t4  \
0     1         CA  0.011719     1  -5.28545  26.775801  -9.527310 -12.793200   
1     1         CA  0.015625     1  -2.42842  28.430901  -9.323510  -3.757230   
2     1         CA  0.019531     1  10.67150  30.420200  15.350700  24.724001   
3     1         CA  0.023438     1  11.45250  25.609800   2.433080  12.412500   
4     1         CA  0.027344     1   7.28321  25.942600   0.113564   5.748000   

      eeg_t6     eeg_t5  ...     eeg_c4     eeg_p4    eeg_poz   eeg_c3  \
0  16.717800  33.737499  ...  37.368999  17.437599  19.201900  20.5968   
1  15.969300  30.443600  ...  31.170799  19.399700  19.689501  21.3547   
2  16.143101  32.142799  ... -12.012600  19.396299  23.171700  22.4076   
3  20.533300  31.494101  ...  18.574100  23.156401  22.641199  19.3367   
4  19.833599  28.753599  ...   6.555440  22.754700  22.670300  20.2932   

    eeg_cz     eeg_o2     ecg           r         gsr 

In [4]:
float_cols = df.select_dtypes(include=['float64']).columns
df[float_cols] = df[float_cols].astype('float32')

int_cols = df.select_dtypes(include=['int64']).columns
df[int_cols] = df[int_cols].apply(pd.to_numeric, downcast='integer')

print(df.dtypes.value_counts())
print('memory:', df.memory_usage(deep=True).sum() / 1e9, 'GB')

float32    24
int8        2
object      2
Name: count, dtype: int64
memory: 0.968616911 GB


In [5]:
# EDA — experiment durations
duration = df.groupby('experiment')['time'].agg(['min', 'max'])
duration['duration_seconds'] = duration['max'] - duration['min']
duration['duration_minutes'] = duration['duration_seconds'] / 60
print(duration)

                 min         max  duration_seconds  duration_minutes
experiment                                                          
CA          0.003000  360.066406        360.063416          6.001057
DA          0.007812  360.371094        360.363281          6.006055
SS          0.003000  360.218750        360.215759          6.003596


In [6]:
# EDA — missing values
missing = df.isnull().sum()
print(missing[missing > 0])
print('total missing values', df.isnull().sum().sum())

Series([], dtype: int64)
total missing values 0


In [7]:
# EDA — class distribution
print(df['event'].value_counts())
print('\nIn percentage')
print(df['event'].value_counts(normalize=True) * 100)

event
A    2848809
C    1652686
D     235329
B     130597
Name: count, dtype: int64

In percentage
event
A    58.528099
C    33.954038
D     4.834778
B     2.683084
Name: proportion, dtype: float64


In [8]:
# EDA — first 10 seconds (C appears at 29% from the start — do NOT drop)
early_rows = df[df['time'] < 10]
print(early_rows['event'].value_counts(normalize=True) * 100)

event
A    70.824118
C    29.175882
Name: proportion, dtype: float64


In [9]:
# ── FEATURE ENGINEERING 1: Rolling mean + std  (same as original) ─────────────
signals_columns = [col for col in df.columns
                   if col not in ['time', 'experiment', 'seat', 'crew', 'event']]
windows = [256, 1280, 2560]  # 1s, 5s, 10s at 256 Hz

for col in tqdm(signals_columns, desc='rolling mean/std'):
    grouped = df.groupby(['experiment', 'crew', 'seat'])[col]
    for window in windows:
        df[f'{col}_rolling_mean_{window}'] = grouped.transform(
            lambda x: x.rolling(window, min_periods=1).mean()
        ).astype('float32')
        df[f'{col}_rolling_std_{window}'] = grouped.transform(
            lambda x: x.rolling(window, min_periods=1).std()
        ).astype('float32')

rolling_cols = [c for c in df.columns if 'rolling' in c]
df[rolling_cols] = df[rolling_cols].fillna(df[rolling_cols].mean())

print('NaNs after fillna:', df[rolling_cols].isnull().sum().sum())
print(df.shape)  # expect (4867421, 166)

rolling mean/std:  52%|█████▏    | 12/23 [01:21<01:06,  6.08s/it]C:\Users\mhimi\AppData\Local\Temp\ipykernel_22824\2261153204.py:12: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[f'{col}_rolling_std_{window}'] = grouped.transform(
C:\Users\mhimi\AppData\Local\Temp\ipykernel_22824\2261153204.py:9: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[f'{col}_rolling_mean_{window}'] = grouped.transform(
C:\Users\mhimi\AppData\Local\Temp\ipykernel_22824\2261153204.py:12: PerformanceWarning: DataFrame is highly fragmented.  This is us

NaNs after fillna: 0
(4867421, 166)


In [10]:
# ── FEATURE ENGINEERING 2: Rolling range (max - min)  [NEW] ──────────────────
# Captures amplitude spikes — Class B (Startle) shows sudden extreme values
# that std misses but range catches directly.
# 23 signals x 3 windows = 69 new features
for col in tqdm(signals_columns, desc='rolling range'):
    grouped = df.groupby(['experiment', 'crew', 'seat'])[col]
    for window in windows:
        col_max = grouped.transform(lambda x: x.rolling(window, min_periods=1).max())
        col_min = grouped.transform(lambda x: x.rolling(window, min_periods=1).min())
        df[f'{col}_rolling_range_{window}'] = (col_max - col_min).astype('float32')

range_cols = [c for c in df.columns if 'rolling_range' in c]
print('NaNs in range cols:', df[range_cols].isnull().sum().sum())
print(df.shape)  # expect (4867421, 235)

rolling range:   0%|          | 0/23 [00:00<?, ?it/s]C:\Users\mhimi\AppData\Local\Temp\ipykernel_22824\1711656475.py:10: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[f'{col}_rolling_range_{window}'] = (col_max - col_min).astype('float32')
C:\Users\mhimi\AppData\Local\Temp\ipykernel_22824\1711656475.py:10: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[f'{col}_rolling_range_{window}'] = (col_max - col_min).astype('float32')
C:\Users\mhimi\AppData\Local\Temp\ipykernel_22824\1711656475.py:10: PerformanceWarning: DataFrame is 

NaNs in range cols: 0
(4867421, 235)


In [11]:
# ── FEATURE ENGINEERING 3: First differences  [NEW] ──────────────────────────
# diff[t] = signal[t] - signal[t-1] = instantaneous rate of change.
# Startle is a sudden physiological jump — the first derivative captures when.
# 23 signals = 23 new features
for col in tqdm(signals_columns, desc='first differences'):
    df[f'{col}_diff1'] = (
        df.groupby(['experiment', 'crew', 'seat'])[col]
        .diff()
        .fillna(0.0)
        .astype('float32')
    )

diff_cols = [c for c in df.columns if c.endswith('_diff1')]
print('NaNs in diff cols:', df[diff_cols].isnull().sum().sum())
print(df.shape)  # expect (4867421, 258)

first differences:   0%|          | 0/23 [00:00<?, ?it/s]C:\Users\mhimi\AppData\Local\Temp\ipykernel_22824\4034121012.py:6: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[f'{col}_diff1'] = (
first differences:   4%|▍         | 1/23 [00:00<00:19,  1.10it/s]C:\Users\mhimi\AppData\Local\Temp\ipykernel_22824\4034121012.py:6: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[f'{col}_diff1'] = (
first differences:   9%|▊         | 2/23 [00:01<00:18,  1.13it/s]C:\Users\mhimi\AppData\Local\Temp\ipykernel_22824\4034121012.py:6: Performa

NaNs in diff cols: 0
(4867421, 258)


In [12]:
# ── MEMORY MANAGEMENT: save, delete, reload ───────────────────────────────────
# Frees groupby internals held in RAM after rolling steps.
# No kernel restart needed — equivalent effect.
df.to_parquet('checkpoint_features_v2.parquet', index=False)
print('Saved checkpoint_features_v2.parquet —', df.shape)

del df
gc.collect()

df = pd.read_parquet('checkpoint_features_v2.parquet')
for col in df.select_dtypes('float64').columns:
    df[col] = df[col].astype('float32')
print('Reloaded clean:', df.shape, '  memory:', df.memory_usage(deep=True).sum() / 1e9, 'GB')

Saved checkpoint_features_v2.parquet — (4867421, 258)


KeyboardInterrupt: 

In [ ]:
# ── FEATURE ENGINEERING 4: EEG frequency band power  [NEW] ───────────────────
# EEG is analysed in frequency domain in neuroscience.
# Each band maps to a known cognitive state:
#   Delta (0.5-4 Hz)  : drowsiness
#   Theta (4-8 Hz)    : memory load / fatigue
#   Alpha (8-13 Hz)   : relaxed alertness  <- dominant in Baseline (A)
#   Beta  (13-30 Hz)  : active thinking / stress
#   Gamma (30-45 Hz)  : higher cognition
# 20 EEG channels x 5 bands = 100 new features

FS = 256
BANDS = {
    'delta': (0.5,  4.0),
    'theta': (4.0,  8.0),
    'alpha': (8.0, 13.0),
    'beta' : (13.0, 30.0),
    'gamma': (30.0, 45.0),
}
POWER_WINDOW = 1280  # 5 seconds at 256 Hz

SOS = {name: butter(4, [lo, hi], btype='band', fs=FS, output='sos')
       for name, (lo, hi) in BANDS.items()}

eeg_cols = [c for c in df.columns
            if c.startswith('eeg_')
            and 'rolling' not in c
            and 'diff' not in c
            and 'range' not in c]
print('EEG channels:', len(eeg_cols))

groups_index = df.groupby(['experiment', 'crew', 'seat']).groups

for eeg_col in tqdm(eeg_cols, desc='EEG band power'):
    for band_name, sos in SOS.items():
        feat_name = f'{eeg_col}_{band_name}_power'
        out = np.zeros(len(df), dtype=np.float32)
        for idx in groups_index.values():
            signal = df.loc[idx, eeg_col].values.astype(np.float64)
            filtered = sosfiltfilt(sos, signal)
            inst_power = filtered ** 2
            smoothed = (pd.Series(inst_power)
                        .rolling(POWER_WINDOW, min_periods=1)
                        .mean().values)
            out[idx] = smoothed.astype(np.float32)
        df[feat_name] = out

band_cols = [c for c in df.columns if any(b in c for b in BANDS)]
print('NaNs in band-power cols:', df[band_cols].isnull().sum().sum())
print(df.shape)  # expect (4867421, 358)

EEG channels: 20


EEG band power:  95%|█████████▌| 19/20 [00:47<00:02,  2.40s/it]C:\Users\mhimi\AppData\Local\Temp\ipykernel_13856\170950370.py:45: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[feat_name] = out
C:\Users\mhimi\AppData\Local\Temp\ipykernel_13856\170950370.py:45: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[feat_name] = out
C:\Users\mhimi\AppData\Local\Temp\ipykernel_13856\170950370.py:45: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor perfo

NaNs in band-power cols: 0
(4867421, 358)


In [ ]:
# ── DATA PREPARATION: Per-subject normalization  (same as original) ───────────
# Each pilot has their own biological baseline.
# Normalising per (crew, seat) makes deviations relative to that pilot's own normal.
cols_to_normalize = [col for col in df.columns
                     if col not in ['time', 'event', 'crew', 'experiment', 'seat']]
print(len(cols_to_normalize))  # expect 353

for (crew, seat), idx in tqdm(df.groupby(['crew', 'seat']).groups.items(),
                              desc='normalising pilots'):
    pilot_data = df.loc[idx, cols_to_normalize]
    pilot_std  = pilot_data.std().replace(0, 1)
    df.loc[idx, cols_to_normalize] = (pilot_data - pilot_data.mean()) / pilot_std

float_cols = df.select_dtypes('float64').columns
df[float_cols] = df[float_cols].astype('float32')

print(df.shape)
print('NaNs after normalisation:', df[cols_to_normalize].isnull().sum().sum())

353


normalising pilots:  94%|█████████▍| 17/18 [19:09<01:07, 67.61s/it]


KeyboardInterrupt: 

In [ ]:
# ── BUILD X / y / groups  (same logic as original) ───────────────────────────
# Extract groups and y BEFORE dropping columns.
groups = df['crew'].values

le = LabelEncoder()
y  = le.fit_transform(df['event'].values)  # A=0, B=1, C=2, D=3

cols_to_drop  = ['experiment', 'crew', 'seat', 'time', 'event']
feature_names = [c for c in df.columns if c not in cols_to_drop]
X = df.drop(columns=cols_to_drop).values.astype('float32')

print('X shape:', X.shape)      # expect (4867421, 353)
print('y shape:', y.shape)
print('y classes:', np.unique(y))
print('groups:', np.unique(groups))
print('feature_names:', len(feature_names))

np.save('X_v2.npy',      X)
np.save('y_v2.npy',      y)
np.save('groups_v2.npy', groups)

# save feature names so last cell can load them
with open('feature_names_v2.json', 'w') as f:
    json.dump(feature_names, f)

del df
gc.collect()
print('df deleted — memory freed')

X shape: (4867421, 353)
y shape: (4867421,)
y classes: [0 1 2 3]
groups: [ 1  2  3  4  5  6  7  8 13]
feature_names: 353
df deleted — memory freed


In [ ]:
# per-crew class distribution (same as original)
for crew_id in sorted(np.unique(groups)):
    mask = groups == crew_id
    dist = pd.Series(y[mask]).value_counts(normalize=True).sort_index() * 100
    dist = dist.reindex([0, 1, 2, 3], fill_value=0.0)
    print(f'Crew {crew_id}: A={dist[0]:5.1f}%  B={dist[1]:5.1f}%  C={dist[2]:5.1f}%  D={dist[3]:5.1f}%')

Crew 1: A= 52.7%  B=  1.7%  C= 40.4%  D=  5.2%
Crew 2: A= 59.1%  B=  2.8%  C= 33.3%  D=  4.9%
Crew 3: A= 59.2%  B=  2.8%  C= 33.3%  D=  4.7%
Crew 4: A= 59.4%  B=  2.8%  C= 33.3%  D=  4.5%
Crew 5: A= 59.0%  B=  2.8%  C= 33.3%  D=  4.9%
Crew 6: A= 59.3%  B=  2.8%  C= 33.3%  D=  4.7%
Crew 7: A= 59.3%  B=  2.8%  C= 33.3%  D=  4.6%
Crew 8: A= 58.9%  B=  2.8%  C= 33.5%  D=  4.8%
Crew 13: A= 58.7%  B=  2.8%  C= 33.3%  D=  5.2%


In [ ]:
# ── LOGO-CV with tuned LightGBM ───────────────────────────────────────────────
# Changes vs original:
#   n_estimators=2000 + early_stopping(50)  was default 100
#   learning_rate=0.05                       was default 0.1
#   num_leaves=127                           was default 31
#   min_child_samples=100                    was default 20
#   subsample=0.8, colsample_bytree=0.7      row/feature bagging
#   reg_alpha=0.1, reg_lambda=1.0            L1 + L2 regularisation

logo   = LeaveOneGroupOut()
scores = []
cv_results = {}
importances_per_fold = []

RESULTS_FILE = 'cv_results_v2.json'
if os.path.exists(RESULTS_FILE):
    with open(RESULTS_FILE) as f:
        cv_results = json.load(f)
    print(f'Resuming from {len(cv_results)} completed folds')

for fold, (train_idx, val_idx) in enumerate(logo.split(X, y, groups)):
    fold_key = f'fold_{fold + 1}'
    crew_id  = int(groups[val_idx][0])

    if fold_key in cv_results:
        scores.append(cv_results[fold_key]['log_loss'])
        print(f'Fold {fold+1} — crew {crew_id} — skipped (done): {cv_results[fold_key]["log_loss"]:.4f}')
        continue

    X_train, X_val = X[train_idx], X[val_idx]
    y_train, y_val = y[train_idx], y[val_idx]

    model = lgb.LGBMClassifier(
        objective         = 'multiclass',
        num_class         = 4,
        n_estimators      = 2000,
        learning_rate     = 0.05,
        num_leaves        = 127,
        min_child_samples = 100,
        subsample         = 0.8,
        subsample_freq    = 1,
        colsample_bytree  = 0.7,
        reg_alpha         = 0.1,
        reg_lambda        = 1.0,
        is_unbalance      = True,
        random_state      = 42,
        n_jobs            = -1,
        verbose           = -1,
    )
    model.fit(
        X_train, y_train,
        eval_set  = [(X_val, y_val)],
        callbacks = [
            lgb.early_stopping(stopping_rounds=50, verbose=False),
            lgb.log_evaluation(period=-1),
        ],
    )

    importances_per_fold.append(model.feature_importances_)

    val_preds  = model.predict_proba(X_val)
    fold_score = log_loss(y_val, val_preds)
    scores.append(fold_score)
    cv_results[fold_key] = {
        'crew'          : crew_id,
        'log_loss'      : fold_score,
        'best_iteration': model.best_iteration_,
    }
    print(f'Fold {fold+1} — crew {crew_id} — log loss: {fold_score:.4f}  (best iter: {model.best_iteration_})')

    with open(RESULTS_FILE, 'w') as f:
        json.dump(cv_results, f, indent=2)

    del X_train, X_val, model
    gc.collect()

print(f'\nMean log loss: {np.mean(scores):.4f}')
print(f'Std:           {np.std(scores):.4f}')

JSONDecodeError: Expecting value: line 1 column 1 (char 0)

In [ ]:
# comparison with original results
v1_file = 'cv_results.json'
if os.path.exists(v1_file):
    with open(v1_file) as f:
        v1 = json.load(f)
    v1_scores = [v['log_loss'] for v in v1.values()]

    print(f'{"Fold":>6}  {"Crew":>6}  {"v1 LL":>8}  {"v2 LL":>8}  {"Delta":>8}')
    print('-' * 48)
    for (fk, rv2), rv1 in zip(cv_results.items(), v1.values()):
        delta = rv2['log_loss'] - rv1['log_loss']
        tag   = 'better' if delta < 0 else ''
        print(f'{fk:>6}  {rv2["crew"]:>6}  {rv1["log_loss"]:>8.4f}  {rv2["log_loss"]:>8.4f}  {delta:>+8.4f}  {tag}')
    print('-' * 48)
    v2m = np.mean(scores)
    v1m = np.mean(v1_scores)
    print(f'{"MEAN":>6}  {"":>6}  {v1m:>8.4f}  {v2m:>8.4f}  {v2m - v1m:>+8.4f}')

In [ ]:
# feature importances
# feature_names was saved in the build-X cell above
with open('feature_names_v2.json') as f:
    feature_names = json.load(f)

if importances_per_fold:
    mean_importance   = np.mean(importances_per_fold, axis=0)
    importance_series = pd.Series(mean_importance, index=feature_names).sort_values(ascending=False)
    print('Top 20 features:')
    print(importance_series.head(20))
    importance_series.to_csv('feature_importances_v2.csv')
    print('Saved feature_importances_v2.csv')

# fast version — subsample 30% of data, 10 trials
rng = np.random.default_rng(42)
sample_idx = rng.choice(len(X), size=int(len(X) * 0.3), replace=False)
X_s = X[sample_idx]
y_s = y[sample_idx]
g_s = groups[sample_idx]

def objective(trial):
    params = dict(
        objective         = 'multiclass',
        num_class         = 4,
        n_estimators      = 2000,
        learning_rate     = trial.suggest_float('learning_rate', 0.01, 0.1, log=True),
        num_leaves        = trial.suggest_int('num_leaves', 63, 255),
        min_child_samples = trial.suggest_int('min_child_samples', 50, 200),
        colsample_bytree  = trial.suggest_float('colsample_bytree', 0.5, 0.9),
        reg_alpha         = trial.suggest_float('reg_alpha', 0.0, 1.0),
        reg_lambda        = trial.suggest_float('reg_lambda', 0.0, 2.0),
        is_unbalance      = True,
        random_state      = 42,
        n_jobs            = -1,
        verbose           = -1,
    )
    logo, scores = LeaveOneGroupOut(), []
    for train_idx, val_idx in logo.split(X_s, y_s, g_s):
        model = lgb.LGBMClassifier(**params)
        model.fit(X_s[train_idx], y_s[train_idx],
                  eval_set=[(X_s[val_idx], y_s[val_idx])],
                  callbacks=[lgb.early_stopping(30, verbose=False),
                              lgb.log_evaluation(-1)])
        scores.append(log_loss(y_s[val_idx], model.predict_proba(X_s[val_idx])))
    return np.mean(scores)

study = optuna.create_study(direction='minimize')
study.optimize(objective, n_trials=10)

print('Best score:', study.best_value)
print('Best params:', study.best_params)